In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import mph
import mphsweepkit as msk

In [4]:
# Start the COMSOL client
client = mph.start()

In [5]:
# Load the model
model = client.load('cuboid_capacitor_solved.mph')

Load an already solved CascadedSweepModel

In [6]:
csm = msk.CascadedSweepModel(model, 'Cuboid Capacitor Study', show_param_names=True)

Initialized CascadedSweepModel
Study name: Cuboid Capacitor Study
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'height', 'length', 'width'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.sw1'
        - Excitation Sweep (Parametric) -> 'voltage', 'core_temperature'
          - Coil Geometry Analysis (CoilCurrentCalculation)
            - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Coil Geometry Analysis', 'Frequency Sweep']
Loop lengths: [5, 3, 1, 1, 5]
--------------------------------
Data updated from MPh-model.
Input data shape: (75, 7)
Reset output data to shape of the input data: (75, 0)
Combined shape: (75, 7)


Post-processing of global equations

In [7]:
# Load the customized post-processing expressions from a JSON file
post_processing_exprs = msk.load_post_processing_exprs("global_post_processing_expressions.json", print_info=False)

# Perform post-processing of global data
csm.post_process_globals(post_processing_exprs)

# Show the first few rows of the combined data
csm.combined_data.head()

name,height,length,width,matsw.comp1.sw1,voltage,core_temperature,freq,Z,C
unit,mm,mm,mm,,V,,Hz,Ohm,F
group,Geometry Sweep,Geometry Sweep,Geometry Sweep,Material Sweep,Excitation Sweep,Excitation Sweep,Frequency Sweep,post-processing,post-processing
0,10.0,20.0,2.0,1.0,10.0,60.0,100000.0,173.887303-329.878642j,3.775569e-09-1.990197e-09j
1,10.0,20.0,2.0,1.0,10.0,60.0,200000.0,86.579277-202.074587j,3.327239e-09-1.425563e-09j
2,10.0,20.0,2.0,1.0,10.0,60.0,300000.0,57.901198-146.616024j,3.130219e-09-1.236178e-09j
3,10.0,20.0,2.0,1.0,10.0,60.0,400000.0,44.149661-115.505044j,3.005636e-09-1.148849e-09j
4,10.0,20.0,2.0,1.0,10.0,60.0,500000.0,36.242485- 95.376007j,2.916314e-09-1.108187e-09j


In [8]:
# Save the input and output dataframes to CSV files
csm.save_global_data()

Saved result data to: X:\Till_data\repositories\MPhSweepKit\examples\studies\ferrite_plate_capacitor\global_data


Post-processing of fields on a selected domain

In [9]:
# All available selections can be printed with
csm.print_available_selections()

# Select a domain selection for post-processing.
# The selection name must exist in the COMSOL model:
selection_name = "cross-section"
selection_type = "bnd"
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type=selection_type)
print(f"Selected domain tag: {selection_domain_tag}")

# Choose a name for the dataset that will be created from the selection:
selection_dataset_name = "Solution Cross-Section"

# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)

# A dataset can be removed with
# csm.node_datasets.children()[-1].remove()

Available selections:
geom1_csel1_pnt cross-section
geom1_csel1_edg cross-section
geom1_csel1_bnd cross-section
geom1_csel1_dom cross-section
Selected domain tag: geom1_csel1_bnd
Creating dataset 'Solution Cross-Section' from selection 'cross-section' of type 'bnd'.


In [13]:
# Evaluate field expressions on the selection dataset and export the results to text files.
#   For each geometry configuration, a separate text file is created with the corresponding results.
#   This allows mesh reuse for expression evaluation and straightforward result comparison.

# The loop levels for the COMSOL export can be obtained with
comsol_looplevels = csm.get_geometry_looplevels()

# Load customized post-processing expressions
post_processing_exprs = msk.load_post_processing_exprs("fields_post_processing_expressions.json", print_info=False)

# Convert the post-processing expressions into lists for the COMSOL export
expressions = [entry["expression"] for entry in post_processing_exprs.values()]
labels = [entry["label"] for entry in post_processing_exprs.values()]
units = [entry["unit"] for entry in post_processing_exprs.values()]

# TODO: Add a visualization of the different geometries

print(f"Loop levels for COMSOL export: {comsol_looplevels}\n")

# Iterate over the loop levels and export the results for each geometry configuration
for geometry_no, looplevel_arr in enumerate(comsol_looplevels):

    export_node = csm.export_dataset_with_expressions(
        dataset_name=selection_dataset_name,
        expressions=expressions,
        descriptions=labels,
        units=units,
        looplevel=looplevel_arr,
        looplevelinput=['manual'] * len(looplevel_arr),
        geometry_no=geometry_no
    )

    print(f"Loop level: {export_node.property('looplevel')}\n")

Loop levels from COMSOL: [[1, 2, 3, 4, 5], [1, 2, 3], [1], [1], [1, 2, 3, 4, 5]]
Sweep configuration is valid.
Loop levels for COMSOL export: [[[1, 2, 3, 4, 5], [1], [1, 2, 3]], [[1, 2, 3, 4, 5], [1], [4, 5, 6]], [[1, 2, 3, 4, 5], [1], [7, 8, 9]], [[1, 2, 3, 4, 5], [1], [10, 11, 12]], [[1, 2, 3, 4, 5], [1], [13, 14, 15]]]

Dataset to be exported:             'Solution Cross-Section'
Overwrite existing export node:     'Expressions on Solution Cross-Section'
List of expressions:                ['mef.normB', 'mef.normE']
Loop level: [array([1., 2., 3., 4., 5.]) array([1.]) array([1., 2., 3.])]

Dataset to be exported:             'Solution Cross-Section'
Overwrite existing export node:     'Expressions on Solution Cross-Section'
List of expressions:                ['mef.normB', 'mef.normE']
Loop level: [array([1., 2., 3., 4., 5.]) array([1.]) array([4., 5., 6.])]

Dataset to be exported:             'Solution Cross-Section'
Overwrite existing export node:     'Expressions on Solution Cro

Save post-processing results

Release Model

In [14]:
client.remove(model)
client.clear()
client.disconnect()